In [ ]:
def build_micro_features(df:pd.DataFrame):
    """
    Intra ticker features
    """
    df = df.copy()
    # Returns 
    df["log_ret_1"] = np.log(df["close"] / df["close"].shift(1))

    for w in [3, 5, 10, 20]:
        df[f"log_ret_{w}"] = np.log(df["close"] / df["close"].shift(w))
        # df[f"ret_mean_{w}"] = df["log_ret_1"].rolling(w).mean() too correlated with above!
    df["ret_mean_5"] = df["log_ret_1"].rolling(5).mean()
    # Volatility
    for w in [5, 10, 20]:
        df[f"vol_{w}"] = df["log_ret_1"].rolling(w).std()

    # volatility ratios (regime indicators)
    df["vol_ratio_5_20"] = df["vol_5"] / df["vol_20"]

    # ATR (Average True Range)
    high_low = df["high"] - df["low"]
    high_close = np.abs(df["high"] - df["close"].shift(1))
    low_close = np.abs(df["low"] - df["close"].shift(1))

    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["atr_14"] = true_range.rolling(14).mean()
    df["atr_pct"] = df["atr_14"] / df["close"]

    # TREND & Momentum strength
    # Moving averages & ratios
    for w in [5, 10, 20, 50]:
        df[f"sma_{w}"] = df["close"].rolling(w).mean()
        df[f"ema_{w}"] = df["close"].ewm(span=w, adjust=False).mean()

    df["sma_ratio_5_20"] = df["sma_5"] / df["sma_20"] - 1
    df["sma_ratio_10_50"] = df["sma_10"] / df["sma_50"] - 1
    df["ema_ratio_5_20"] = df["ema_5"] / df["ema_20"] - 1

    # Slope 
    for w in [10, 20]:
        df[f"slope_{w}"] = rolling_slope(df["close"], w)
    
    # Distance to extremes
    for w in [10, 20]:
        df[f"dist_high_{w}"] = df["close"] / df["high"].rolling(w).max() - 1
        df[f"dist_low_{w}"] = df["close"] / df["low"].rolling(w).min() - 1

    # Oscillators: RSI
    # df["rsi_7"] = rsi(df["close"], 7)
    df["rsi_14"] = rsi(df["close"], 14)

    # Stochastic oscillator 
    low_14 = df["low"].rolling(14).min()
    high_14 = df["high"].rolling(14).max()
    df["stoch_k"] = 100 * (df["close"] - low_14) / (high_14 - low_14)
    #df["stoch_d"] = df["stoch_k"].rolling(3).mean()

    # williams r
    #df["williams_r"] = -100 * (high_14 - df["close"]) / (high_14 - low_14)

    # volume liquidity 
    #df["log_volume"] = np.log(df["volume"])

    for w in [5, 20]:
        df[f"volu_mean_{w}"] = df["volume"].rolling(w).mean()
        df[f"volu_ratio_{w}"] = df["volume"] / df[f"volu_mean_{w}"]

    # On-Balance Volume non stationary
    # df["obv"] = (np.sign(df["log_ret_1"]).fillna(0) * df["volume"]).cumsum()

    # Volume-return interaction
    df["volu_ret_1"] = df["log_ret_1"] * df["volu_ratio_5"]

    # candle structure (price action) LESS USEFUL FOR HORIZON 7-9
    df["body"] = (df["close"] - df["open"]).abs() / df["open"]

    df["upper_wick"] = (df["high"] - df[["close", "open"]].max(axis=1)) / df["open"]
    df["lower_wick"] = (df[["close", "open"]].min(axis=1) - df["low"]) / df["open"]

    df["true_range_pct"] = true_range / df["close"]
    df["gap"] = (df["open"] - df["close"].shift(1)) / df["close"].shift(1)

    return df


In [ ]:
def build_macro_features(df: pd.DataFrame, df_macro: pd.DataFrame):
    """
    Macro ticker features
    """
    df = df.copy()
    
    ibex = df_macro[df_macro["ticker"] == "^IBEX"]
    sp = df_macro[df_macro["ticker"] == "^GSPC"]
    vix = df_macro[df_macro["ticker"] == "^VIX"]
    # ALIGN SERIES 

    df = df.merge(
    ibex[["date", "close"]].rename(columns={"close": "ibx_close"}),
    on="date",
    how="left"
    )
    
    df = df.merge(
    sp[["date", "close"]].rename(columns={"close": "sp_close"}),
    on="date",
    how="left"
    )
    
    df = df.merge(
    vix[["date", "close"]].rename(columns={"close": "vix_close"}),
    on="date",
    how="left"
    )

    # Drop later !!!
    df[f"ibx_log_ret_1"] = np.log(df["ibx_close"] / df["ibx_close"].shift(1))
    df[f"sp_log_ret_1"] = np.log(df["sp_close"] / df["sp_close"].shift(1))
    
    # Returns 
    for w in [5, 10, 20]:
        df[f"ibx_log_ret_{w}"] = np.log(df["ibx_close"] / df["ibx_close"].shift(w))
    for w in [20,50]:
        df[f"sp_log_ret_{w}"] = np.log(df["sp_close"] / df["sp_close"].shift(w))

    # Volatility
    for w in [10,20,60]:
        df[f"ibx_vol_{w}"] = df["ibx_log_ret_1"].rolling(w).std()
    for w in [20,60,100]:
        df[f"sp_vol_{w}"] = df["sp_log_ret_1"].rolling(w).std()

    # Volatilty ratio
    df["ibx_vol_ratio_10_60"] = df["ibx_vol_10"] / df["ibx_vol_60"]
    df["sp_vol_ratio_20_100"] = df["sp_vol_20"] / df["sp_vol_100"]
    
    df["vix_chg_1"] = df["vix_close"].pct_change()
    # Shock detector
    df["vix_chg_z_5"] = df["vix_chg_1"] / df["vix_chg_1"].rolling(5).std()
    
    # Medium-term stress regime
    df["vix_dev_20"] = (
        df["vix_close"] - df["vix_close"].rolling(20).mean()
    )

    # Long-term volatility regime (stable)
    df["vix_pctile_250"] = (
        df["vix_close"]
        .rolling(250)
        .apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1])
    )

    # Relative to market
    df["rel_ret_5"] = df["log_ret_5"] - df["ibx_log_ret_5"]
    df["rel_ret_20"] = df["log_ret_20"] - df["ibx_log_ret_20"]
    df["rel_vol_20"] = df["vol_20"] / df["ibx_vol_20"]

    return df
  

A metamodel can:

Detect market regime (bull / bear / volatile)

Then choose which base model to trust

Example:

Model A works in trending markets

Model B works in mean-reverting markets

Metamodel predicts regime → selects model

This is powerful in non-stationary environments like finance.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)


param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [5, 7, 10],
    "max_features": ["sqrt", 0.5],
    "min_samples_leaf": [1, 5, 10],

}

model = RandomForestClassifier()

search = GridSearchCV(
    model,
    param_grid,
    cv=tscv,
    scoring="accuracy",
)